# Liu2024 TWFB — Classifier Playground

This notebook loads the **precomputed covariance matrices** and runs a classifier on them.
It reproduces the per-subject motor-imagery decoding numbers without touching the raw EEG.

### How to use it (3 steps)
1. **Set `COV_DIR`** in the next cell to the covariance folder (the one containing the `sub-XX_cov.npz` files).
2. **Pick a classifier** in the "Choose your classifier" cell (FgMDM is set up and works out of the box).
3. **Run all cells** (Kernel → Restart & Run All).

> **You do _not_ need the raw EEG dataset.** The covariances already contain everything the
> classifier needs, so the covariance folder is the only path you have to set.

### Requirements
`pip install -r requirements.txt`


## 1. The only setting you must edit

In [ ]:
# ============================================================
#   EDIT THIS ONE LINE  v v v
# ============================================================
COV_DIR = "/path/to/cov_90b378e769"     # folder containing the sub-XX_cov.npz files

# ---- evaluation settings (match the paper's 60/40, 10-repeat setup) ----
SUBJECTS_LIMIT = 5       # quick check on the first 5 subjects. Set to None for all 50 (the real run).
TEST_FRAC      = 0.40    # 40% of trials held out for testing each repeat (paper: 60/40 split)
N_REPEATS      = 10      # random train/test splits per subject (paper: 10 repeats)
INNER_FOLDS    = 5       # inner CV folds for honest band/window selection (not a paper setting; 3 is faster)
RUN_LEAKY      = True    # also compute the "leaky" number, for comparison (see notes at the bottom)
RANDOM_STATE   = 2026


## 2. Choose your classifier

`make_classifier()` returns a fresh classifier each time it's called.
**To try a different method, just change what this function returns** — a few ready-made
options are listed underneath.

In [ ]:
import os, glob
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from pyriemann.classification import FgMDM

# ============================================================
#   PICK YOUR CLASSIFIER HERE  v v v
#   FgMDM == the paper's DGFMDRM (Riemannian classifier), with NO dimension reduction.
# ============================================================
def make_classifier():
    return FgMDM(metric="riemann")


# ---- Other options: replace the body of make_classifier() above with one of these ----
#
# (a) TSLDA baseline  (tangent-space projection -> LDA):
#     from pyriemann.tangentspace import TangentSpace
#     from sklearn.pipeline import make_pipeline
#     from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
#     return make_pipeline(TangentSpace(metric="riemann"), LinearDiscriminantAnalysis())
#
# (b) Plain MDRM  (minimum distance to Riemannian mean):
#     from pyriemann.classification import MDM
#     return MDM(metric="riemann")
#
# (c) FULL TWFB+DGFMDRM  (paper Table 3: LTSA-style SPD reduction -> FgMDM).
#     Paste the SpdReducer class below into this cell first, then use:
#     from sklearn.pipeline import make_pipeline
#     return make_pipeline(SpdReducer(n_components=6), FgMDM(metric="riemann"))
#
#     from sklearn.base import BaseEstimator, TransformerMixin
#     class SpdReducer(BaseEstimator, TransformerMixin):
#         """Reduce each covariance to a smaller SPD matrix; W is fit on TRAIN only."""
#         def __init__(self, n_components=6): self.n_components = n_components
#         def fit(self, X, y=None):
#             _, V = np.linalg.eigh(X.mean(axis=0))
#             self.W_ = V[:, ::-1][:, :self.n_components]
#             return self
#         def transform(self, X):
#             return np.einsum("ci,ncd,dj->nij", self.W_, X, self.W_)


## 3. Load the covariances *(no need to edit)*

Each `sub-XX_cov.npz` file holds, for one subject, a covariance matrix **per trial** for
every band x time-window combination (the `tw__...` arrays), plus the labels `y`.

In [ ]:
def load_subjects(cov_dir):
    files = sorted(glob.glob(os.path.join(cov_dir, "sub-*_cov.npz")))
    if not files:
        raise FileNotFoundError(f"No 'sub-*_cov.npz' files found in:\n  {cov_dir}")
    subjects = []
    for f in files:
        z = np.load(f, allow_pickle=False)
        # the 'tw__{lo}_{hi}__{start}' arrays are the time-window x frequency-band views
        cells = {k: z[k] for k in z.files if k.startswith("tw__")}
        y     = z["y"].astype(int)
        sid   = int(os.path.basename(f).split("-")[1][:2])
        subjects.append({"sid": sid, "cells": cells, "y": y})
    return subjects

SUBJECTS = load_subjects(COV_DIR)
if SUBJECTS_LIMIT:
    SUBJECTS = SUBJECTS[:SUBJECTS_LIMIT]

# quick sanity print
ex   = SUBJECTS[0]
view = next(iter(ex["cells"].values()))
print(f"Loaded {len(SUBJECTS)} subject(s) from:\n  {COV_DIR}\n")
print(f"Each subject has {len(ex['cells'])} band x window covariance views")
print(f"Each view has shape {view.shape}  ->  (trials, channels, channels)")
print(f"Labels for sub-{ex['sid']:02d}: {ex['y']}   (0 = left hand, 1 = right hand)")


## 4. Evaluation *(no need to edit)*

For each subject we run repeated train/test splits and report **balanced accuracy** (chance = 50%;
for the balanced left/right classes here this is essentially the paper's accuracy):

- **honest** — the best band/window is chosen using *only the training trials* (inner cross-validation). This is the fair number.
- **leaky** — the best band/window is chosen by looking at the *test* trials. Shown only for comparison.

In [ ]:
def _score(Xtr, ytr, Xte, yte):
    """Fit a fresh classifier on train covariances, return balanced accuracy on test."""
    clf = make_classifier()
    clf.fit(Xtr, ytr)
    return balanced_accuracy_score(yte, clf.predict(Xte))

def evaluate_subject(cells, y):
    names = list(cells.keys())
    outer = list(StratifiedShuffleSplit(n_splits=N_REPEATS, test_size=TEST_FRAC,
                                        random_state=RANDOM_STATE).split(np.zeros(len(y)), y))
    honest, leaky = [], []
    for tr, te in outer:
        # ---- HONEST: pick the band/window using ONLY training trials ----
        inner = list(StratifiedKFold(INNER_FOLDS, shuffle=True,
                                     random_state=RANDOM_STATE).split(np.zeros(len(tr)), y[tr]))
        inner_scores = []
        for n in names:
            C = cells[n]
            inner_scores.append(np.mean([_score(C[tr[a]], y[tr[a]], C[tr[b]], y[tr[b]])
                                         for a, b in inner]))
        best = names[int(np.argmax(inner_scores))]
        C = cells[best]
        honest.append(_score(C[tr], y[tr], C[te], y[te]))

        # ---- LEAKY: pick the band/window using the TEST trials (max-on-test) ----
        if RUN_LEAKY:
            leaky.append(max(_score(cells[n][tr], y[tr], cells[n][te], y[te]) for n in names))

    return float(np.mean(honest)), (float(np.mean(leaky)) if RUN_LEAKY else np.nan)


## 5. Run

> **Heads up on runtime.** With `N_REPEATS = 10` and all 50 subjects this is a long run — it refits
> the classifier for every band/window on every split. Keep `SUBJECTS_LIMIT` small while iterating
> and set it to `None` for the full paper-style run (overnight). Setting `INNER_FOLDS = 3` speeds the
> honest protocol up with almost no change in the result.

In [ ]:
rows = []
for s in SUBJECTS:
    h, l = evaluate_subject(s["cells"], s["y"])
    rows.append({"subject": s["sid"], "honest_bacc": h, "leaky_bacc": l})
    line = f"sub-{s['sid']:02d}:  honest = {h*100:5.1f}%"
    if RUN_LEAKY:
        line += f"   |   leaky = {l*100:5.1f}%"
    print(line)

df = pd.DataFrame(rows)
print("-" * 48)
print(f"HONEST  mean = {df['honest_bacc'].mean()*100:5.2f}%      (chance = 50%)")
if RUN_LEAKY:
    print(f"LEAKY   mean = {df['leaky_bacc'].mean()*100:5.2f}%")
    print(f"GAP (leaky - honest) = {(df['leaky_bacc'].mean()-df['honest_bacc'].mean())*100:.1f} pts")
df


## Notes

**honest vs leaky.** With many band/window combinations to choose from, picking the one that
scores best *on the test set* inflates accuracy a lot. The honest number selects that choice on
training data only. The gap between them is the thing to watch.

**What "good" looks like.** On this dataset the honest balanced accuracy tends to land near chance
(~50-52%) for the Riemannian pipeline, while the leaky number is much higher. That contrast is the
point of the comparison.

**Switching covariance sets.** Point `COV_DIR` at a different `cov_...` folder to use a different
preprocessing/grid (e.g. the 8-band vs 19-band sets). The notebook adapts automatically — it just
reads however many band/window views are inside the files.

**The three methods (so the names don't trip you up).**
`FgMDM` alone = **DGFMDRM** (no reduction). `TangentSpace + LDA` = the **TSLDA** baseline.
`SpdReducer -> FgMDM` = the full **TWFB+DGFMDRM** (the reduction is paper Table 3, step 4).
Note: the paper's "LTSA" (Local Tangent Space Alignment, a dimensionality reducer) is **not** the
same as TSLDA's Riemannian "tangent space" — same words, different algorithms.
